In [2]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
import requests
import time
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import PandasTools
import math

---
## 1. Load Raw Data
Load the CSV that was saved after querying the ChEMBL API.
> If you haven't downloaded it yet, run the data download cell in the original notebook first.

In [ ]:
activity = new_client.activity
query = activity.filter(
    target_chembl_id='CHEMBL203',
    type='IC50',
    assay_type='B'
)

all_data = []
# The ChEMBL API has shown sensitivity to large requests, so we'll fetch data in small pages.
# We will use a smaller limit (e.g., 5 or 10) and implement retries.
limit = 10       # Initial page size
offset = 0
max_retries = 3   # Number of retries for each page request

print("Starting data retrieval with pagination...")

while True:
    page_data = None
    success = False
    current_page = [] # Initialize current_page for scope
    for attempt in range(max_retries):
        print(f"  Attempting to retrieve records from offset {offset} to {offset + limit} (attempt {attempt + 1}/{max_retries})...")
        try:
            # Use slicing for pagination, which the chembl_webresource_client supports
            current_page = query[offset : offset + limit]
            if current_page:
                all_data.extend(current_page)
                print(f"  Successfully retrieved {len(current_page)} records. Total accumulated: {len(all_data)}")
                success = True
                break
            else:
                # If current_page is empty, there's no more data
                print("  No more data found for this offset range. Ending retrieval.")
                success = True # Consider it a success for loop termination
                break
        except Exception as e:
            print(f"  Error encountered for offset {offset}: {e}")
            if attempt < max_retries - 1:
                wait_time = 5 * (attempt + 1) # Exponential backoff
                print(f"  Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"  Max retries reached for offset {offset}. Skipping this block.")
                # If even after retries we fail, we might want to log this and potentially stop or adjust strategy
                break

    if not success or not current_page: # If we failed to get data after retries or no data was found
        break

    offset += limit
    time.sleep(1) # Pause between successful page requests to be gentle on the API

print(f"Total records obtained: {len(all_data)}")

df = pd.DataFrame.from_records(all_data)
print(df.head())

Solicitando offset 0 (límite 100)...
  Obtenidos 100 registros. Total: 100
Solicitando offset 100 (límite 100)...
  Obtenidos 100 registros. Total: 200
Solicitando offset 200 (límite 100)...
Error 500: <!doctype html>
<html lang="en" class="vf-no-js">
  <head>
    <script>
// Detect if JS is on and swap vf-no-js for vf-js on the html element
(function(H){H.className=H.className.replace(/\bvf-no-js\b
Total registros obtenidos: 200


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,32260,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.041
1,None,None,32267,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,0.17
2,None,None,32680,[],CHEMBL677833,In vitro inhibition of Epidermal growth factor...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,9.3
3,None,None,32770,[],CHEMBL674643,Inhibitory concentration of EGF dependent auto...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,500.0
4,None,None,32772,[],CHEMBL674643,Inhibitory concentration of EGF dependent auto...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,None,None,IC50,uM,UO_0000065,None,3000.0


In [ ]:
df.to_csv('egfr_bioactivity_raw.csv', index=False)

In [4]:
df.columns

Index(['action_type', 'activity_comment', 'activity_id', 'activity_properties',
       'assay_chembl_id', 'assay_description', 'assay_type',
       'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint',
       'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment',
       'data_validity_description', 'document_chembl_id', 'document_journal',
       'document_year', 'ligand_efficiency', 'molecule_chembl_id',
       'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value',
       'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id',
       'standard_flag', 'standard_relation', 'standard_text_value',
       'standard_type', 'standard_units', 'standard_upper_value',
       'standard_value', 'target_chembl_id', 'target_organism',
       'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type',
       'units', 'uo_units', 'upper_value', 'value'],
      dtype='object')

---
## 2. Quality Filters
Three filters are applied in sequence:

| Filter | Reason |
|---|---|
| `pchembl_value` not null | ChEMBL-curated pIC50. Only exists for IC50/Ki/EC50 — drops `% Inhibition` rows automatically |
| `potential_duplicate == 0` | ChEMBL flags records likely reported in multiple sources from the same experiment |
| `standard_relation == '='` | Removes censored values (`>` or `<`) where the true potency is unknown |

In [5]:
print(f'Before filters: {len(df):,} records')

# Filter 1: must have a curated pChEMBL value
df = df[df['pchembl_value'].notnull()]
print(f'After pchembl_value filter    : {len(df):,}')

# Filter 2: remove ChEMBL-flagged potential duplicates
df = df[df['potential_duplicate'] == 0]
print(f'After potential_duplicate filter: {len(df):,}')

# Filter 3: keep only exact measurements (relation = '=')
# Note: the API returns '=' without quotes; the web-UI CSV adds single quotes.
# The .query() approach handles both cases cleanly.
df = df[df['standard_relation'].isin(['=', "'='"]) ]
print(f'After standard_relation filter  : {len(df):,}')

Before filters: 23,120 records
After pchembl_value filter    : 16,815
After potential_duplicate filter: 15,192
After standard_relation filter  : 15,192


---
## 3. Deduplicate Molecules with InChI Keys

**Why InChI keys instead of canonical SMILES?**
The same molecule can arrive with different SMILES strings from different papers
(different atom ordering, with/without explicit stereochemistry, as a salt, etc.).
An InChI key is a fixed 27-character hash of the molecular graph — two SMILES pointing
to the same molecule always produce the same InChI key.

**How duplicates are handled:**
- `zero_sd` group (all measurements agree): keep the first row unchanged
- `nonzero_sd` group (measurements differ): replace the pChEMBL value with the group mean

This is equivalent to what ChEMBL itself does when computing its aggregated `pchembl_value`.

In [6]:
# ── Function definitions ──────────────────────────────────────────────────────

def identify_duplicates(df, smiles_column):
    """
    Standardises SMILES, computes InChI keys, and flags duplicate molecules.
    Returns an annotated copy of df with columns:
        canonical_smiles, inchi_key, is_valid, is_duplicate,
        duplicate_group, occurrence_count
    """

    def standardize_smiles(smi):
        """Re-canonicalise a SMILES string using RDKit."""
        try:
            mol = Chem.MolFromSmiles(smi)
            return Chem.MolToSmiles(mol, canonical=True) if mol else None
        except Exception:
            return None

    def calculate_inchi_key(smi):
        """Convert SMILES to an InChI key (molecule fingerprint, writing-invariant)."""
        try:
            mol = Chem.MolFromSmiles(smi)
            return Chem.MolToInchiKey(mol) if mol else None
        except Exception:
            return None

    out = df.copy()
    out['canonical_smiles']  = out[smiles_column].apply(standardize_smiles)
    out['inchi_key']         = out['canonical_smiles'].apply(calculate_inchi_key)
    out['is_valid']          = out['canonical_smiles'].notna()
    out['is_duplicate']      = out['inchi_key'].duplicated(keep='first')
    out['duplicate_group']   = out.groupby('inchi_key').ngroup()
    counts                   = out['inchi_key'].value_counts()
    out['occurrence_count']  = out['inchi_key'].map(counts)
    return out


def get_duplicate_summary(df_analysis):
    """Print a summary of the duplicate analysis."""
    summary = {
        'total_records'            : len(df_analysis),
        'invalid_smiles'           : (~df_analysis['is_valid']).sum(),
        'unique_molecules'         : df_analysis['inchi_key'].nunique(),
        'duplicate_records_removed': df_analysis['is_duplicate'].sum(),
        'molecules_with_duplicates': (df_analysis['occurrence_count'] > 1).sum(),
        'max_duplicates_one_mol'   : int(df_analysis['occurrence_count'].max()),
    }
    return summary


def process_duplicates(df, inchi_key_col, pchembl_col):
    """
    Collapses duplicate molecules to a single row per InChI key.
    - zero_sd  (all IC50 values agree) : keep first row as-is
    - nonzero_sd (IC50 values differ)  : keep first row, replace pChEMBL with group mean
    - single_entry                     : no change
    Returns (final_df, summary_df).
    """
    df_work = df.copy()

    # Group statistics per unique molecule
    stats = (
        df_work.groupby(inchi_key_col)[pchembl_col]
        .agg(['count', 'mean', 'std'])
        .round(4)
        .reset_index()
    )
    stats.columns = [inchi_key_col, 'count', 'mean', 'std']

    processed = []

    for _, row in stats.iterrows():
        group = df_work[df_work[inchi_key_col] == row[inchi_key_col]].copy()
        rep   = group.iloc[[0]].copy()   # representative row

        if row['count'] == 1:
            rep['group_type'] = 'single_entry'
        elif row['std'] < 0.01:          # identical measurements
            rep['group_type'] = 'zero_sd'
        else:                            # variable measurements → use mean
            rep[pchembl_col]  = row['mean']
            rep['group_type'] = 'nonzero_sd'

        processed.append(rep)

    final = pd.concat(processed).sort_values(inchi_key_col).reset_index(drop=True)

    summary = pd.DataFrame([
        ('initial_records',          len(df_work)),
        ('unique_molecules',         len(stats)),
        ('single_entry',             (stats['count'] == 1).sum()),
        ('duplicate_groups',         (stats['count'] > 1).sum()),
        ('  → zero_sd (identical)',  (stats.query('count > 1')['std'] < 0.01).sum()),
        ('  → nonzero_sd (averaged)',(stats.query('count > 1')['std'] >= 0.01).sum()),
        ('final_records',            len(final)),
    ], columns=['step', 'count']).set_index('step')

    return final, summary


print('Functions defined.')

Functions defined.


In [7]:
# ── Step 1: Identify duplicates and invalid SMILES ───────────────────────────
df_analysis = identify_duplicates(df, smiles_column='canonical_smiles')

summary = get_duplicate_summary(df_analysis)
print('Duplicate analysis:')
for k, v in summary.items():
    print(f'  {k:<35}: {v:,}')

Duplicate analysis:
  total_records                      : 15,192
  invalid_smiles                     : 17
  unique_molecules                   : 9,876
  duplicate_records_removed          : 5,315
  molecules_with_duplicates          : 8,205
  max_duplicates_one_mol             : 86


In [8]:
# ── Step 2: Drop invalid SMILES before processing ────────────────────────────
n_invalid = (~df_analysis['is_valid']).sum()
if n_invalid > 0:
    print(f'Dropping {n_invalid} rows with invalid SMILES')
    df_analysis = df_analysis[df_analysis['is_valid']].copy()
else:
    print('No invalid SMILES found — all good.')

Dropping 17 rows with invalid SMILES


In [9]:
# ── Step 3: Collapse duplicates ──────────────────────────────────────────────
final_df, summary_df = process_duplicates(
    df_analysis,
    inchi_key_col='inchi_key',
    pchembl_col='pchembl_value'
)

print('Deduplication summary:')
print(summary_df.to_string())

Deduplication summary:
                           count
step                            
initial_records            15175
unique_molecules            9876
single_entry                6970
duplicate_groups            2906
  → zero_sd (identical)      286
  → nonzero_sd (averaged)   2620
final_records               9876


---
## 4. Assign Bioactivity Classes

Cutoffs aligned with **DeepEGFR (Malik et al., 2025)** — the benchmark paper for this target:

| Class | pChEMBL | IC50 |
|---|---|---|
| Active | ≥ 6.0 | ≤ 1,000 nM |
| Intermediate | 5.0 – 6.0 | 1,000 – 10,000 nM |
| Inactive | ≤ 5.0 | ≥ 10,000 nM |

In [10]:
def assign_class(pchembl):
    if pchembl >= 6.0:
        return 'active'
    elif pchembl <= 5.0:
        return 'inactive'
    else:
        return 'intermediate'

final_df['bioactivity_class'] = final_df['pchembl_value'].apply(assign_class)

print('Class distribution:')
print(final_df['bioactivity_class'].value_counts().to_string())

Class distribution:
bioactivity_class
active          7439
intermediate    1722
inactive         715


---
## 5. Save Outputs

In [12]:
# ── Full 3-class dataset ─────────────────────────────────────────────────────
cols_to_keep = [
    'molecule_chembl_id',
    'canonical_smiles',
    'inchi_key',
    'pchembl_value',
    'bioactivity_class',
    'group_type',        # audit trail: single_entry / zero_sd / nonzero_sd
]
final_df = final_df[cols_to_keep].reset_index(drop=True)

In [13]:
final_df

,molecule_chembl_id,canonical_smiles,inchi_key,pchembl_value,bioactivity_class,group_type
0,CHEMBL268868,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA-UHFFFAOYSA-N,6.5200,active,single_entry
1,CHEMBL2029442,COc1cc(N2CCN(C)CC2)ccc1Nc1ncc2c(n1)N(c1cccc(N)...,AAFHSECTTHOVFV-UHFFFAOYSA-N,6.3200,active,single_entry
2,CHEMBL2048906,CC(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc45)c(Cl...,AAGKMGNYUYCEPD-UHFFFAOYSA-N,8.5400,active,single_entry
3,CHEMBL5757238,C=CC(=O)Nc1cc(Nc2nccc(-c3cnc4c(ccn4C)c3)n2)c(O...,AAGOVSWSOCZTHE-UHFFFAOYSA-N,8.5050,active,nonzero_sd
4,CHEMBL1240554,CCOc1ccc(-c2nn(C3CCCC3)c3ncnc(N)c23)cc1OC,AAHKGRWRYBCWDL-UHFFFAOYSA-N,5.8500,intermediate,single_entry
...,...,...,...,...,...,...
9871,CHEMBL247710,NC1CCN(Cc2ccn3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c...,ZZRGSHVUZJRFKX-UHFFFAOYSA-N,7.4200,active,single_entry
9872,CHEMBL2178349,CN1CCN(c2ccc(Nc3ncc4nc(Nc5ccccc5)n(C)c4n3)cc2)CC1,ZZSYRUGQSJWOAC-UHFFFAOYSA-N,7.5733,active,nonzero_sd
9873,CHEMBL4570293,C=CC(=O)Nc1cc(Nc2nccc(NC(=O)c3cccc4ccccc34)n2)...,ZZVIASMIOWABNB-UHFFFAOYSA-N,7.4800,active,single_entry
9874,CHEMBL2437476,C=CC(=O)Nc1cccc(Nc2ncc3ncc(=O)n(-c4ccc(OC)cc4)...,ZZYRBJVUKKZJCS-UHFFFAOYSA-N,5.7967,intermediate,nonzero_sd


In [19]:
# ── Save ─────────────────────────────────────────────────────────────────────
final_df.to_csv('/workspaces/Drug_discovery/data/processed/egfr_bioactivity_cleaned.csv', index=False)

print(f'Saved egfr_bioactivity_cleaned.csv  → {len(final_df):,} compounds (3 classes)')

final_df.head(5)

Saved egfr_bioactivity_cleaned.csv  → 9,876 compounds (3 classes)
Saved egfr_bioactivity_2class.csv   → 8,154 compounds (active + inactive only)


,molecule_chembl_id,canonical_smiles,inchi_key,pchembl_value,bioactivity_class,group_type
0,CHEMBL268868,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA-UHFFFAOYSA-N,6.520,active,single_entry
1,CHEMBL2029442,COc1cc(N2CCN(C)CC2)ccc1Nc1ncc2c(n1)N(c1cccc(N)...,AAFHSECTTHOVFV-UHFFFAOYSA-N,6.320,active,single_entry
2,CHEMBL2048906,CC(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc45)c(Cl...,AAGKMGNYUYCEPD-UHFFFAOYSA-N,8.540,active,single_entry
3,CHEMBL5757238,C=CC(=O)Nc1cc(Nc2nccc(-c3cnc4c(ccn4C)c3)n2)c(O...,AAGOVSWSOCZTHE-UHFFFAOYSA-N,8.505,active,nonzero_sd
4,CHEMBL1240554,CCOc1ccc(-c2nn(C3CCCC3)c3ncnc(N)c23)cc1OC,AAHKGRWRYBCWDL-UHFFFAOYSA-N,5.850,intermediate,single_entry
